## Data Ingestion


In [6]:
import langchain
from typing import List, Dict , Any
import pandas as pd
import tiktoken

In [7]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

print("Setup completed!")

Setup completed!


## Understanding Document Structure In Langchain

In [8]:
## Create a simple document
doc=Document(
    page_content="The is the main text content that will be embedded and searched.",
    metadata={
        "source":"example.txt",
        "page":1,
        "author":"Ashish",
        "date_created":"2026-06-18",
        "custom_feild":"any_value"
        
        
    }
)
print("Document Structure")
print(f"content:{doc.page_content}")
print(f"content:{doc.metadata}")

Document Structure
content:The is the main text content that will be embedded and searched.
content:{'source': 'example.txt', 'page': 1, 'author': 'Ashish', 'date_created': '2026-06-18', 'custom_feild': 'any_value'}


## Text Files(.txt) - The Simplest Case
{#2-text-files}

In [9]:
## Create a simple txt File
import os
os.makedirs("data/text_files",exist_ok=True)

In [10]:
sample_texts = {
    "data/text_files/python_intro.txt": """Python Programming Introduction

Python is a high-level, interpreted, object-oriented, and general-purpose programming language created by Guido van Rossum in 1991.

Python is known for its simple syntax, readability, and versatility. It is widely used in web development, data science, artificial intelligence, machine learning, automation, cybersecurity, and software development.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support.
""",
    "data/text_files/machine_learning_intro.txt": """Machine Learning Introduction

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.

It focuses on developing algorithms that can identify patterns in data and make predictions or decisions based on that data.
"""
}
for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)
print("sample text files created!")        
        

sample text files created!


## TextLoader-Read Single File

In [11]:

from langchain_community.document_loaders import TextLoader

## Loading  a single text file
loader=TextLoader("data/text_files/python_intro.txt", encoding="utf-8")


documents=loader.load()
print(f" Loaded{len(documents)}(documents")
print(f"content preview:{documents[0].page_content[:100]}...")
print(f"Metadata:{documents[0].metadata}")



 Loaded1(documents
content preview:Python Programming Introduction

Python is a high-level, interpreted, object-oriented, and general-p...
Metadata:{'source': 'data/text_files/python_intro.txt'}


## DirectoryLoader - Multiple Text Files

In [12]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Load all text files from directory
dir_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)

documents = dir_loader.load()

print(f"📁 Loaded {len(documents)} documents")

for i, doc in enumerate(documents):
    print(f"\n📄 Document {i+1}:")
    print(f"Source: {doc.metadata['source']}")
    print(f"Length: {len(doc.page_content)} characters")

100%|██████████| 2/2 [00:00<00:00, 496.98it/s]

📁 Loaded 2 documents

📄 Document 1:
Source: data\text_files\machine_learning_intro.txt
Length: 308 characters

📄 Document 2:
Source: data\text_files\python_intro.txt
Length: 510 characters


## Text Splitting Stratgies

In [13]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)
print(documents)

[Document(metadata={'source': 'data\\text_files\\machine_learning_intro.txt'}, page_content='Machine Learning Introduction\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.\n\nIt focuses on developing algorithms that can identify patterns in data and make predictions or decisions based on that data.\n'), Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted, object-oriented, and general-purpose programming language created by Guido van Rossum in 1991.\n\nPython is known for its simple syntax, readability, and versatility. It is widely used in web development, data science, artificial intelligence, machine learning, automation, cybersecurity, and software development.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong communi

In [14]:
## Method 1 - Characters Text Splitter
text=documents[0].page_content
text

'Machine Learning Introduction\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.\n\nIt focuses on developing algorithms that can identify patterns in data and make predictions or decisions based on that data.\n'

In [15]:
# Method 1 - Character Text Splitter

print("1. CHARACTER TEXT SPLITTER")

char_splitter = CharacterTextSplitter(
    separator="\n",          # Split on new line
    chunk_size=200,          # Maximum chunk size in characters
    chunk_overlap=20,        # Overlap between chunks
    length_function=len      # How to measure chunk size
)

char_chunks = char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

1. CHARACTER TEXT SPLITTER
Created 2 chunks
First chunk: Machine Learning Introduction
Machine learning is a subset of artificial intelligence that enables s...


In [16]:
print(char_chunks[0])
print("---------")
print(char_chunks[1])
print("------------")
if len(char_chunks) > 2:
    print(char_chunks[2])
else:
    print("No third chunk available; only", len(char_chunks), "chunks.")

Machine Learning Introduction
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.
---------
It focuses on developing algorithms that can identify patterns in data and make predictions or decisions based on that data.
------------
No third chunk available; only 2 chunks.


In [17]:
## Method 2: Recursive character splitting (Recommended)
print("\n2️⃣ RECURSIVE CHARACTER TEXT SPLITTER")
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],  # Changed separators
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)
recursive_chunks = recursive_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")


2️⃣ RECURSIVE CHARACTER TEXT SPLITTER
Created 2 chunks
First chunk: Machine Learning Introduction
Machine learning is a subset of artificial intelligence that enables s...


In [18]:
print(recursive_chunks[0])
print("---------")
print(recursive_chunks[1])
print("------------")
if len(recursive_chunks) > 2:
    print(recursive_chunks[2])
else:
    print("No third chunk available; only", len(recursive_chunks), "chunks.")

Machine Learning Introduction

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.
---------
It focuses on developing algorithms that can identify patterns in data and make predictions or decisions based on that data.
------------
No third chunk available; only 2 chunks.


In [19]:
# Create text without natural break points

simple_text = (
    "This is sentence one and it is quite long. "
    "This is sentence two and it is also quite long. "
    "This is sentence three which is even longer than."
    "This is sentence three which is even longer than."
)

splitter = RecursiveCharacterTextSplitter(
    separators=[" "],  # Only split on spaces
    chunk_size=80,
    chunk_overlap=20,
    length_function=len
)

chunks = splitter.split_text(simple_text)

print(f"\nSimple text example - {len(chunks)} chunks:\n")

for i in range(len(chunks) - 1):
    print(f"Chunk {i+1}: '{chunks[i]}'")
    print(f"Chunk {i+2}: '{chunks[i+1]}'")

print()


Simple text example - 3 chunks:

Chunk 1: 'This is sentence one and it is quite long. This is sentence two and it is also'
Chunk 2: 'two and it is also quite long. This is sentence three which is even longer'
Chunk 2: 'two and it is also quite long. This is sentence three which is even longer'
Chunk 3: 'is even longer than.This is sentence three which is even longer than.'



In [20]:
from langchain_text_splitters import TokenTextSplitter

# Method 3: Token-based splitting
print("\n 3️⃣ TOKEN TEXT SPLITTER")
token_splitter = TokenTextSplitter(
    chunk_size=50,
    chunk_overlap=10 
)
token_chunks = token_splitter.split_text(simple_text)
print(f"Created {len(token_chunks)} chunks")
print(f"First chunk: {token_chunks[0][:100]}...")


 3️⃣ TOKEN TEXT SPLITTER
Created 1 chunks
First chunk: This is sentence one and it is quite long. This is sentence two and it is also quite long. This is s...


In [21]:
# 📊 Comparison

print("\n📊 Text Splitting Methods Comparison:")

print("\nCharacterTextSplitter:")
print("    ✅ Simple and predictable")
print("    ✅ Good for structured text")
print("    ❌ May break mid-sentence")
print("    Use when: Text has clear delimiters")

print("\nRecursiveCharacterTextSplitter:")
print("    ✅ Respects text structure")
print("    ✅ Tries multiple separators")
print("    ✅ Best general-purpose splitter")
print("    ❌ Slightly more complex")
print("    Use when: Default choice for most texts")

print("\nTokenTextSplitter:")
print("    ✅ Respects model token limits")
print("    ✅ More accurate for embeddings")
print("    ❌ Slower than character-based")
print("    Use when: Working with token-limited models")


📊 Text Splitting Methods Comparison:

CharacterTextSplitter:
    ✅ Simple and predictable
    ✅ Good for structured text
    ❌ May break mid-sentence
    Use when: Text has clear delimiters

RecursiveCharacterTextSplitter:
    ✅ Respects text structure
    ✅ Tries multiple separators
    ✅ Best general-purpose splitter
    ❌ Slightly more complex
    Use when: Default choice for most texts

TokenTextSplitter:
    ✅ Respects model token limits
    ✅ More accurate for embeddings
    ❌ Slower than character-based
    Use when: Working with token-limited models
